In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "engine").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from leagues.nba import NBA
from engine.carryover import fit_carryover

DATA = ROOT / "data" / "processed"
games = pd.read_csv(DATA / "games.csv", parse_dates=["date"])

print("ROOT: ", ROOT)
print("games:", len(games), "rows,", games["season"].nunique(), "seasons")

ROOT:  c:\Projects\FuturesTakeHomeAssignmentPP
games: 27065 rows, 21 seasons


In [2]:
from engine.observations import split_at_asof, build_observations
from engine.ratings import fit_ratings

regular = games[games["season_type"] == "Regular Season"]

# A season is treated as atypical when teams did not play a full schedule.
# Lockout and pandemic seasons are shorter, were played under different
# conditions, and the off-seasons either side of them are not normal ones.
# Deriving the flag from the schedule itself avoids a hardcoded list that would
# silently go stale the next time the league does something unusual.
games_per_team = (
    pd.concat([
        regular[["season", "home_team"]].rename(columns={"home_team": "team"}),
        regular[["season", "away_team"]].rename(columns={"away_team": "team"}),
    ])
    .groupby(["season", "team"]).size()
    .groupby("season").median()
)

end_ratings = {}
rows = []

for season in sorted(regular["season"].unique()):
    season_games = regular[regular["season"] == season]
    asof = season_games["date"].max() + pd.Timedelta(days=1)

    played, remaining = split_at_asof(games, season=int(season), asof=asof)
    assert len(remaining) == 0, f"{season}: {len(remaining)} games left unplayed"

    fit = fit_ratings(build_observations(played))
    end_ratings[int(season)] = fit.ratings

    rows.append({
        "season": int(season),
        "games": len(played),
        "per_team": int(games_per_team[season]),
        "teams": fit.ratings.size,
        "home_adv": round(fit.home_advantage, 2),
        "margin_sd": round(fit.residual_sd, 2),
        "spread": round(float(fit.ratings.max() - fit.ratings.min()), 1),
        "best": fit.ratings.index[0],
    })

season_table = pd.DataFrame(rows)
season_table["atypical"] = season_table["per_team"] != 82
print(season_table.to_string(index=False))
print("\natypical:", sorted(season_table.loc[season_table["atypical"], "season"]))

 season  games  per_team  teams  home_adv  margin_sd  spread best  atypical
   2006   1230        82     30      3.10      10.67    14.1  SAS     False
   2007   1230        82     30      2.79      11.05    11.8  SAS     False
   2008   1230        82     30      3.16      10.74    16.4  BOS     False
   2009   1230        82     30      3.08      10.58    16.2  CLE     False
   2010   1230        82     30      2.58      10.91    15.5  ORL     False
   2011   1230        82     30      2.91      10.33    14.7  MIA     False
   2012    990        66     30      2.52      11.06    19.4  SAS      True
   2013   1229        82     30      2.98      10.90    16.8  OKC     False
   2014   1230        82     30      2.38      10.83    17.0  SAS     False
   2015   1230        82     30      2.17      10.96    18.2  GSW     False
   2016   1230        82     30      2.41      10.72    18.6  SAS     False
   2017   1230        82     30      2.77      11.54    16.7  GSW     False
   2018   12

In [3]:
from engine.carryover import fit_carryover, prior_weight

# Seasons flagged atypical upstream are excluded from both sides of every pair.
# A 66-game lockout season and a bubble season measure strength under different
# conditions, and the off-season either side of them is not a normal one.
skip = set(season_table.loc[season_table["atypical"], "season"])

carryover = fit_carryover(end_ratings, skip_seasons=skip)
print(carryover)

# The noise scale used to convert the width of the prior into a weight is a
# property of the league, not of one season, so it is taken across seasons.
margin_sd = float(season_table.loc[~season_table["atypical"], "margin_sd"].median())

print(f"league margin sd: {margin_sd:.2f}")
print(f"prior weight:     {prior_weight(margin_sd, carryover):.2f}")
print(f"equivalent games: {prior_weight(margin_sd, carryover) ** 2:.1f}")

Carryover(beta=0.6187559667169816, tau=3.470296033342554, n_pairs=447)
league margin sd: 11.01
prior weight:     3.17
equivalent games: 10.1


In [4]:
from itertools import product

from engine.weights import observation_weights
from engine.motivation import decided_flags
from engine.probabilities import schedule_win_probabilities

HALF_LIVES = [None, 180, 120, 90, 60]
DISCOUNTS = [1.0, 0.7, 0.5, 0.3, 0.1]
CUT = "03-01"


def log_loss(p, y):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    y = np.asarray(y, dtype=float)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())


prepared = []
for season in season_table.loc[~season_table["atypical"], "season"]:
    regular = games[(games["season"] == season) & (games["season_type"] == "Regular Season")]
    playoffs = games[(games["season"] == season) & (games["season_type"] == "Playoffs")]
    if playoffs.empty:
        continue

    obs_full = build_observations(regular)
    flags = decided_flags(regular, NBA).set_index("game_id")
    aligned = flags.reindex(obs_full["game_id"])
    # A game is discounted if either side had nothing left to play for: one
    # resting team distorts the margin just as much as two.
    mask = (aligned["home_settled"] | aligned["away_settled"]).to_numpy()

    prepared.append({
        "season": int(season),
        "season_end": regular["date"].max(),
        "obs_full": obs_full,
        "mask": mask,
        "playoffs": playoffs,
        "share_discounted": float(mask.mean()),
    })

print("share of regular-season games with a settled team, by season:")
print(pd.Series(
    [p["share_discounted"] for p in prepared],
    index=[p["season"] for p in prepared],
).round(3).to_string())

rows = []
for half_life, discount in product(HALF_LIVES, DISCOUNTS):
    scores = []
    for item in prepared:
        w = observation_weights(
            item["obs_full"], item["season_end"],
            half_life_days=half_life,
            discount_mask=item["mask"],
            discount=discount,
        )
        fit = fit_ratings(item["obs_full"], weights=w)
        p = schedule_win_probabilities(
            item["playoffs"], fit.ratings, fit.home_advantage, fit.residual_sd
        )
        scores.append(log_loss(p, item["playoffs"]["home_win"]))
    rows.append({
        "half_life": half_life if half_life else "flat",
        "discount": discount,
        "playoff_ll": round(float(np.mean(scores)), 5),
    })

grid = pd.DataFrame(rows)
print("\n" + grid.pivot(index="half_life", columns="discount", values="playoff_ll").to_string())

share of regular-season games with a settled team, by season:
2006    0.105
2007    0.107
2008    0.116
2009    0.086
2010    0.101
2011    0.124
2013    0.100
2014    0.104
2015    0.124
2016    0.123
2017    0.098
2018    0.115
2019    0.102
2022    0.090
2023    0.103
2024    0.111
2025    0.115
2026    0.128

discount       0.1      0.3      0.5      0.7      1.0
half_life                                             
60         0.62734  0.62740  0.62807  0.62907  0.63090
90         0.62626  0.62624  0.62662  0.62726  0.62852
120        0.62593  0.62588  0.62614  0.62663  0.62763
180        0.62577  0.62568  0.62584  0.62619  0.62695
flat       0.62590  0.62576  0.62577  0.62590  0.62627


In [5]:
def per_game_loss(p, y):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    y = np.asarray(y, dtype=float)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


def losses(half_life, discount):
    out = []
    for item in prepared:
        w = observation_weights(
            item["obs_full"], item["season_end"],
            half_life_days=half_life, discount_mask=item["mask"], discount=discount,
        )
        fit = fit_ratings(item["obs_full"], weights=w)
        p = schedule_win_probabilities(
            item["playoffs"], fit.ratings, fit.home_advantage, fit.residual_sd
        )
        out.append(per_game_loss(p, item["playoffs"]["home_win"]))
    return np.concatenate(out)


# Holding the half-life fixed isolates the discount: the two settings differ in
# nothing else, so the paired difference measures the discount alone.
for half_life in [180, 120, 90]:
    base = losses(half_life, 1.0)
    disc = losses(half_life, 0.3)
    diff = base - disc
    se = diff.std(ddof=1) / np.sqrt(len(diff))
    print(f"half-life {half_life}: gain {diff.mean():.5f} +/- {se:.5f}, t {diff.mean() / se:.2f}")

half-life 180: gain 0.00128 +/- 0.00112, t 1.15
half-life 120: gain 0.00176 +/- 0.00124, t 1.42
half-life 90: gain 0.00230 +/- 0.00136, t 1.69


In [6]:
STEP = pd.Timedelta(days=14)

from engine.carryover import build_prior, prior_weight


def per_game_loss(p, y):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    y = np.asarray(y, dtype=float)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


# Walk-forward validation. At each as-of date the model sees only what had
# happened by then and is scored on the following two weeks of games. This is
# how the model is actually used, and it is the only scheme that avoids leaking
# later results into an earlier rating. Randomly holding out games would do
# exactly that: a January game's teams would be rated partly from March.
weight = prior_weight(NBA.margin_sd, carryover)
records = []

for season in season_table.loc[~season_table["atypical"], "season"]:
    season = int(season)
    if season - 1 not in end_ratings:
        continue

    regular = games[(games["season"] == season) & (games["season_type"] == "Regular Season")]
    teams = sorted(set(regular["home_team"]) | set(regular["away_team"]))
    prior = build_prior(end_ratings[season - 1], carryover, teams)

    asof = regular["date"].min() + STEP
    while asof < regular["date"].max():
        played, remaining = split_at_asof(games, season=season, asof=asof)
        window = remaining[remaining["date"] < asof + STEP]
        if len(played) >= 30 and len(window) > 0:
            obs = build_observations(played)
            actual = window["home_win"].to_numpy()

            for label, fit in [
                ("with_prior", fit_ratings(obs, prior_ratings=prior, prior_weight=weight)),
                ("no_prior", fit_ratings(obs)),
            ]:
                p = schedule_win_probabilities(
                    window, fit.ratings, fit.home_advantage, fit.residual_sd
                )
                records.append({
                    "season": season,
                    "played_per_team": round(2 * len(played) / len(teams)),
                    "scheme": label,
                    "loss": per_game_loss(p, actual).sum(),
                    "n": len(window),
                })
        asof += STEP

walk = pd.DataFrame(records)

# Baseline: the league's home win rate, carrying no information about who plays.
scored = games[
    games["season"].isin(walk["season"].unique())
    & (games["season_type"] == "Regular Season")
]
base_rate = scored["home_win"].mean()
baseline = per_game_loss(np.full(len(scored), base_rate), scored["home_win"]).mean()

bins = [0, 10, 20, 40, 60, 100]
walk["stage"] = pd.cut(walk["played_per_team"], bins, labels=["1-10", "11-20", "21-40", "41-60", "61+"])

summary = (
    walk.groupby(["stage", "scheme"], observed=True)[["loss", "n"]].sum()
    .assign(log_loss=lambda d: (d["loss"] / d["n"]).round(5))
    .reset_index()
    .pivot(index="stage", columns="scheme", values="log_loss")
)
summary["games"] = walk[walk.scheme == "no_prior"].groupby("stage", observed=True)["n"].sum()
summary["prior_gain"] = (summary["no_prior"] - summary["with_prior"]).round(5)

print(f"baseline (home win rate {base_rate:.3f}): {baseline:.5f}\n")
print(summary.to_string())

baseline (home win rate 0.581): 0.68000

scheme  no_prior  with_prior  games  prior_gain
stage                                          
1-10     0.69223     0.62575   1757     0.06648
11-20    0.61275     0.59989   2718     0.01286
21-40    0.63676     0.62815   4927     0.00861
41-60    0.59930     0.60013   5620    -0.00083
61+      0.59119     0.59291   4169    -0.00172


In [7]:
# Does the prior deserve the weight it gets, or should it fade out? The derived
# weight assumes the prior stays as informative as it was in October, which the
# stage breakdown above suggests is not true late in a season.
rows = []
for factor in [0.5, 1.0, 2.0]:
    sub = []
    for season in season_table.loc[~season_table["atypical"], "season"]:
        season = int(season)
        if season - 1 not in end_ratings:
            continue
        regular = games[(games["season"] == season) & (games["season_type"] == "Regular Season")]
        teams = sorted(set(regular["home_team"]) | set(regular["away_team"]))
        prior = build_prior(end_ratings[season - 1], carryover, teams)

        asof = regular["date"].min() + STEP
        while asof < regular["date"].max():
            played, remaining = split_at_asof(games, season=season, asof=asof)
            window = remaining[remaining["date"] < asof + STEP]
            if len(played) >= 30 and len(window) > 0:
                fit = fit_ratings(
                    build_observations(played),
                    prior_ratings=prior, prior_weight=weight * factor,
                )
                p = schedule_win_probabilities(
                    window, fit.ratings, fit.home_advantage, fit.residual_sd
                )
                sub.append({
                    "played_per_team": round(2 * len(played) / len(teams)),
                    "loss": per_game_loss(p, window["home_win"].to_numpy()).sum(),
                    "n": len(window),
                })
            asof += STEP
    df = pd.DataFrame(sub)
    df["stage"] = pd.cut(df["played_per_team"], bins, labels=["1-10", "11-20", "21-40", "41-60", "61+"])
    g = df.groupby("stage", observed=True)[["loss", "n"]].sum()
    rows.append((f"weight x{factor} ({(weight * factor) ** 2:.0f} games)", (g["loss"] / g["n"]).round(5)))

print(pd.DataFrame({name: series for name, series in rows}).to_string())

       weight x0.5 (3 games)  weight x1.0 (11 games)  weight x2.0 (45 games)
stage                                                                       
1-10                 0.64043                 0.62575                 0.63300
11-20                0.60395                 0.59989                 0.61188
21-40                0.63302                 0.62815                 0.63015
41-60                0.59921                 0.60013                 0.60847
61+                  0.59148                 0.59291                 0.60109


In [8]:
from engine.uncertainty import sample_ratings

# How much does a team's strength actually move within a season? Splitting each
# season in half and fitting both halves separately gives two noisy readings of
# the same team. The spread between them mixes estimation error, which the fit
# quantifies, with genuine change. Subtracting the first leaves the second.
rows = []
for season in season_table.loc[~season_table["atypical"], "season"]:
    season = int(season)
    regular = games[(games["season"] == season) & (games["season_type"] == "Regular Season")]
    dates = regular["date"].sort_values()
    midpoint = dates.iloc[len(dates) // 2]

    first = build_observations(regular[regular["date"] < midpoint])
    second = build_observations(regular[regular["date"] >= midpoint])
    if first.empty or second.empty:
        continue

    fit_a, fit_b = fit_ratings(first), fit_ratings(second)
    common = fit_a.ratings.index.intersection(fit_b.ratings.index)
    difference = (fit_b.ratings[common] - fit_a.ratings[common]).to_numpy()

    # Estimation variance of the difference is the sum of the two halves'.
    est_var = (
        np.diag(fit_a.rating_cov.loc[common, common]).mean()
        + np.diag(fit_b.rating_cov.loc[common, common]).mean()
    )
    gap_days = (
        regular.loc[regular["date"] >= midpoint, "date"].median()
        - regular.loc[regular["date"] < midpoint, "date"].median()
    ).days

    rows.append({
        "season": season,
        "observed_var": float(difference.var(ddof=1)),
        "estimation_var": float(est_var),
        "gap_days": gap_days,
    })

drift = pd.DataFrame(rows)
drift["drift_var"] = (drift["observed_var"] - drift["estimation_var"]).clip(lower=0)
drift["drift_sd"] = np.sqrt(drift["drift_var"])
drift["daily_vol"] = drift["drift_sd"] / np.sqrt(drift["gap_days"])

print(drift.round(3).to_string(index=False))
print(f"\nmedian daily volatility: {drift['daily_vol'].median():.4f} points per sqrt(day)")
print(f"gap in the data:         {drift['gap_days'].median():.0f} days")

 season  observed_var  estimation_var  gap_days  drift_var  drift_sd  daily_vol
   2006         7.389           5.246        87      2.143     1.464      0.157
   2007         6.688           5.646        88      1.042     1.021      0.109
   2008         6.919           5.319        88      1.600     1.265      0.135
   2009         5.555           5.183        87      0.373     0.611      0.065
   2010         8.076           5.459        87      2.617     1.618      0.173
   2011         7.804           4.886        88      2.919     1.708      0.182
   2013         7.373           5.483        87      1.891     1.375      0.147
   2014        11.865           5.272        88      6.593     2.568      0.274
   2015         9.187           5.469        88      3.718     1.928      0.206
   2016         5.476           5.333        89      0.144     0.379      0.040
   2017         9.263           6.095        89      3.168     1.780      0.189
   2018        10.493           5.948   

In [9]:
from engine.uncertainty import strength_paths
from engine.carryover import build_prior, prior_weight
from scipy.stats import norm
ASOF_MD = "01-15"
STEP_DAYS = 7
N_SIMS = 2_000
VOLS = [0.0, 0.10, 0.18, 0.25, 0.35]

weight = prior_weight(NBA.margin_sd, carryover)


def simulate_wins(season, daily_vol, rng):
    """
    Final regular-season win totals under one volatility setting.

    Simulated the same way the pipeline does it, minus the postseason: the test
    is about the width of the win distribution, and the bracket plays no part in
    that.
    """
    asof = pd.Timestamp(f"{season}-{ASOF_MD}")
    played, remaining = split_at_asof(games, season=season, asof=asof)
    if played.empty or remaining.empty:
        return None

    teams = sorted(set(played["home_team"]) | set(played["away_team"]))
    position = {t: i for i, t in enumerate(teams)}

    prior = build_prior(end_ratings[season - 1], carryover, teams)
    fit = fit_ratings(build_observations(played), prior_ratings=prior, prior_weight=weight)

    base = np.zeros(len(teams))
    for game in played.itertuples():
        base[position[game.home_team if game.home_win else game.away_team]] += 1

    home_idx = np.array([position[t] for t in remaining["home_team"]])
    away_idx = np.array([position[t] for t in remaining["away_team"]])
    home_field = (
        (~remaining["neutral_site"].astype(bool)).astype(float).to_numpy()
        if "neutral_site" in remaining.columns
        else np.ones(len(remaining))
    )

    step = ((remaining["date"] - asof).dt.days // STEP_DAYS).to_numpy()
    paths, path_teams = strength_paths(
        fit, N_SIMS, int(step.max()) + 1, rng, daily_vol, STEP_DAYS
    )
    order = [path_teams.index(t) for t in teams]
    paths = paths[:, :, order]

    # Advanced indexing pulls each game's strength at its own point on the walk,
    # without ever materialising the full sims-by-games-by-teams array.
    margins = (
        paths[:, step, home_idx]
        - paths[:, step, away_idx]
        + fit.home_advantage * home_field
    )
    home_wins = rng.random(margins.shape) < norm.cdf(margins / fit.residual_sd)

    wins = np.tile(base, (N_SIMS, 1))
    np.add.at(wins, (slice(None), home_idx), home_wins)
    np.add.at(wins, (slice(None), away_idx), ~home_wins)

    return pd.DataFrame(wins, columns=teams)


# Actual final win totals, the thing the distribution is supposed to contain.
actual_wins = {}
for season in season_table.loc[~season_table["atypical"], "season"]:
    regular = games[(games["season"] == season) & (games["season_type"] == "Regular Season")]
    winners = np.where(regular["home_win"], regular["home_team"], regular["away_team"])
    actual_wins[int(season)] = pd.Series(winners).value_counts()

rows = []
for vol in VOLS:
    z_scores, inside = [], []
    for season in season_table.loc[~season_table["atypical"], "season"]:
        season = int(season)
        if season - 1 not in end_ratings:
            continue
        sims = simulate_wins(season, vol, np.random.default_rng(season))
        if sims is None:
            continue
        for team in sims.columns:
            actual = actual_wins[season].get(team)
            if actual is None:
                continue
            column = sims[team]
            sd = column.std()
            if sd > 0:
                z_scores.append((actual - column.mean()) / sd)
            low, high = column.quantile([0.10, 0.90])
            inside.append(low <= actual <= high)

    z = np.array(z_scores)
    rows.append({
        "daily_vol": vol,
        "z_sd": round(float(z.std(ddof=1)), 3),
        "z_mean": round(float(z.mean()), 3),
        "inside_80": round(float(np.mean(inside)), 3),
        "n": len(z),
    })

print(pd.DataFrame(rows).to_string(index=False))

 daily_vol  z_sd  z_mean  inside_80   n
      0.00 1.229  -0.006      0.757 510
      0.10 1.211  -0.006      0.763 510
      0.18 1.171  -0.005      0.776 510
      0.25 1.125  -0.005      0.796 510
      0.35 1.053  -0.005      0.818 510
